In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import date

In [0]:

SCOPE   = "pnp-secert-scope"   
client_id     = dbutils.secrets.get(SCOPE, "pnp-client-id")
client_secret = dbutils.secrets.get(SCOPE, "pnp-client-secret")
tenant_id     = dbutils.secrets.get(SCOPE, "pnp-tenant-id")


spark.conf.set("fs.azure.account.auth.type.pnpbyedge.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.pnpbyedge.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.pnpbyedge.dfs.core.windows.net",client_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.pnpbyedge.dfs.core.windows.net", client_secret)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.pnpbyedge.dfs.core.windows.net", f"https://login.windows.net/{tenant_id}/oauth2/token")

In [0]:
# df_silver = spark.read.format('parquet').option('inferschema', True).option('header', True).load('abfss://silver@pnpbyedge.dfs.core.windows.net/2026-06-06/*.parquet')


today = date.today().strftime('%Y-%m-%d')
path = f'abfss://silver@pnpbyedge.dfs.core.windows.net/{today}/*.parquet'

# Read all files in today's folder
df_silver = spark.read\
    .format('parquet')\
    .option('header', True)\
    .option('inferSchema', True)\
    .load(path)



In [0]:



in_col = ['product_id', 'price', 'product_id', 'brand_id', 'manufacturer_id', 'currency', 'pack_size', 'country', 'is_promo', 'promo_price', 'activity_log', 'upc']

for c in in_col:
    df_silver = df_silver.withColumn(c, 
                                    when(upper(trim(col(c))).isin(['#N/A', 'N/A', 'NA', 'NULL', '']), 
                                    None
                                     ).otherwise(col(c))
                                    
    )

In [0]:
df_silver = df_silver.withColumn('promo_price',
regexp_replace(
    regexp_replace(
        trim(col("promo_price").cast('string')),
    '£',''),
'GBP',"").cast(DoubleType())              
  )

In [0]:
df_silver = df_silver.withColumn('promo_price',
            when(col("promo_price")<= 0.10, None)\
            .otherwise(col("promo_price"))
             )


In [0]:
df_silver = df_silver.withColumn('effective_price',
    when(
        (col('is_promo') == True) & (col('promo_price').isNotNull()),
        col('promo_price')
    )\
    .when(
        (col('is_promo') == True) & (col('price') > col('promo_price') ),
        col('promo_price')
    )
    .otherwise(col('price'))
)

In [0]:
df_silver = df_silver.filter(col('effective_price').isNotNull())

In [0]:
df_silver = df_silver.withColumn('data_qaulity', 
                when(col('Sku').isNull(), 'sku missing')\
                .when(col('retailer_name').isNull(), 'retailer_name missing')\
                .when(col('effective_price').isNull(), 'missing eff price')\
                .when(col('price').isNull(), 'price missing')\
                .when(col('pack_size').isNull(), 'pack_size missing')\
                .when(col('upc').isNull(), 'upc missing')\
                .when(col('currency').isNull(), 'currency missing')\
                .otherwise("valid")
              )


In [0]:
df_gold = df_silver \
    .withColumn('gold_processed_at', current_timestamp())

In [0]:


    df_gold = df_gold.select(
        # ── Identity ──
        col('retailer_name'),
        col('sku'),
        col('upc'),
        col('product_id'),
        col('product_name'),

        # ── Pricing ──
        col('price'),
        col('is_promo'),
        col('promo_price'),
        col('effective_price'),
        col('currency'),
        col('country'),

        # ── Brand & Category ──
        col('brand_id'),
        col('manufacturer_id'),
        col('main_category'),
        col('category_l1'),
        col('category_l2'),
        col('category_l3'),
        col('category_series'),      # already in Silver

        # ── Product details ──
        col('pack_size'),
        col('product_url'),
        col('activity_log'),
        col("data_qaulity"),
        col("gold_processed_at"),
        col('date')      # needed for partition!
    
    )


In [0]:
df_gold.display()

In [0]:
df_gold.count()

In [0]:
df_gold = df_gold.repartition(1)

In [0]:

# ══════════════════════════════════════════
# STEP 3 — Write with partition + replaceWhere
# (this is the corrected Cell 12 from before)
# ══════════════════════════════════════════
run_date = date.today().strftime('%Y-%m-%d')
gold_path = 'abfss://gold@pnpbyedge.dfs.core.windows.net/prices/'

df_gold.write.format('delta') \
    .mode('overwrite') \
    .option("overwriteSchema", "true") \
    .option("replaceWhere", f"date = '{run_date}'") \
    .partitionBy("date") \
    .save(gold_path)

print(f"✅ Gold written for {run_date}, partitioned by date")



In [0]:
watermark_path = 'abfss://gold@pnpbyedge.dfs.core.windows.net/Watermark/'

silver_row_count = df_silver.count()
gold_row_count   = df_gold.count()

watermark_df = spark.createDataFrame(
    [(run_date, silver_row_count, gold_row_count, "success")],
    ["run_date", "silver_rows", "gold_rows", "status"]
).withColumn("processed_at", current_timestamp())

watermark_df.write.format('delta') \
    .mode('append') \
    .option("mergeSchema", "true") \
    .save(watermark_path)

print(f"✅ Watermark logged for {run_date}")
print(f"   silver_rows: {silver_row_count}")
print(f"   gold_rows:   {gold_row_count}")